# 16 OOD-aware selective diagnosis

This notebook separates calibrated family-classification confidence from train-only feature-space distance. It evaluates probability, margin, distance, a small validation-selected combination, and split conformal prediction on the milestone-13 trajectory-level splits.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = (PROJECT_ROOT / '..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from ood_selective import run_experiment

result = run_experiment(project_root=PROJECT_ROOT, result_root=PROJECT_ROOT / 'results')
summary = result['summary']
print(json.dumps({
    'judgement': summary['judgement'],
    'mechanism_comparison_extrapolation_120s': summary['mechanism_comparison_extrapolation_120s'],
    'conformal_extrapolation_120s': summary['conformal_extrapolation_120s'],
    'ood_detection_proxy': summary['ood_detection_proxy'],
}, ensure_ascii=False, indent=2, default=str))

{
  "judgement": "candidate_for_v1_unknown_policy",
  "mechanism_comparison_extrapolation_120s": [
    {
      "mechanism": "combination",
      "coverage": 0.470873786407767,
      "selective_macro_f1": 0.35041730815979255,
      "selective_balanced_accuracy": 0.8,
      "risk": 0.11016949152542371,
      "reject_rate": 0.529126213592233
    },
    {
      "mechanism": "distance",
      "coverage": 0.3592233009708738,
      "selective_macro_f1": 0.3571428571428571,
      "selective_balanced_accuracy": 1.0,
      "risk": 0.0,
      "reject_rate": 0.6407766990291262
    },
    {
      "mechanism": "forced_family",
      "coverage": 1.0,
      "selective_macro_f1": 0.6659730091313389,
      "selective_balanced_accuracy": 0.8314732142857142,
      "risk": 0.19417475728155337,
      "reject_rate": 0.0
    },
    {
      "mechanism": "margin",
      "coverage": 0.5728155339805825,
      "selective_macro_f1": 0.4916152507514634,
      "selective_balanced_accuracy": 0.8,
      "risk": 0.08181

In [2]:
required = [
    '16_selective_metrics.csv',
    '16_ood_detection_metrics.csv',
    '16_conformal_metrics.csv',
    '16_rejection_by_class.csv',
    '16_risk_coverage.csv',
    '16_decision_policy.json',
    '16_summary.json',
]
figures = [
    '16_risk_coverage.png',
    '16_accepted_vs_rejected_severity_gap.png',
    '16_conformal_set_size_coverage.png',
    '16_mechanism_comparison.png',
    '16_focus_class_reject_rates.png',
]
assert summary['cohort_count'] == 505
assert all(summary['assertions'].values())
assert all((PROJECT_ROOT / 'results' / name).exists() for name in required)
assert all((PROJECT_ROOT / 'results' / 'figures' / name).exists() for name in figures)
metrics = pd.read_csv(PROJECT_ROOT / 'results' / '16_selective_metrics.csv')
risk = pd.read_csv(PROJECT_ROOT / 'results' / '16_risk_coverage.csv')
conformal = pd.read_csv(PROJECT_ROOT / 'results' / '16_conformal_metrics.csv')
ood = pd.read_csv(PROJECT_ROOT / 'results' / '16_ood_detection_metrics.csv')
rejection = pd.read_csv(PROJECT_ROOT / 'results' / '16_rejection_by_class.csv')
assert {'probability', 'margin', 'distance', 'combination', 'forced_family'}.issubset(set(metrics['mechanism']))
assert set(risk['target_coverage'].unique()) == {0.5, 0.6, 0.7, 0.8, 0.9, 1.0}
assert set(conformal['alpha'].unique()) == {0.05, 0.1, 0.2}
assert {'RI', 'LOCAC', 'SLBIC'}.issubset(set(rejection['accident_class']))
assert {'extrapolation_test_vs_random_test', 'large_nearest_severity_gap_proxy'}.issubset(set(ood['proxy_type']))
assert summary['judgement'] in {'candidate_for_v1_unknown_policy', 'data_insufficient_for_reliable_ood_rejector'}
print('FULL OOD-AWARE SELECTIVE DIAGNOSIS PASSED')

FULL OOD-AWARE SELECTIVE DIAGNOSIS PASSED
